# Generate Training Data
### This file creates scores, e.g. the style similarity scores, for training the multi-head cross-encoder
##### Created by w45242hy

## Install dependency

In [ ]:
from pathlib import Path
root_path = Path().resolve().parent.parent
print(f"root_path: {root_path}")

!pip install -r {root_path}/requirements.txt

## Import dependency

In [ ]:
from dataset_av import AVDataset
import os
import pandas as pd
from torch.nn import functional as F
from torch.utils.data import DataLoader
from sentence_transformers import SentenceTransformer

## Global variable

In [ ]:
ROOT_PATH = os.path.dirname(os.path.dirname(os.getcwd()))
CSV_NAME = f"AV_trial.csv"
CSV_PATH = os.path.join(ROOT_PATH, "data", "AV", CSV_NAME)
OUTPUT_CSV_NAME = f"{CSV_NAME.split('.')[0]}_final.{CSV_NAME.split('.')[1]}"
OUTPUT_CSV_PATH = os.path.join(ROOT_PATH, "data", "AV", OUTPUT_CSV_NAME)
MODEL_NAME = f"sentence-transformers/all-MiniLM-L6-v2"
SEPARATION_TOKEN = f"[SEP]"

## Generate style similarity score

### Load data

In [ ]:
print(f"Reading csv file at {CSV_PATH} ...")
    
# Load dataset
dataset = AVDataset(CSV_PATH)
dataloader = DataLoader(dataset, batch_size=8, shuffle=False)  # Larger batch for efficiency

### Load model

In [ ]:
# Load pre-trained sentence transformer
model = SentenceTransformer(MODEL_NAME)

### Generate style similarity score

In [ ]:
author_scores = []
style_similarity_scores = []
all_text_1 = []
all_text_2 = []

for (texts_1, texts_2), labels in dataloader:
    # (texts_1, texts_2) is 1 batch

    # Compute embeddings
    # Shape: [batch_size, feature_num]
    embedding_1 = model.encode(texts_1, convert_to_tensor=True)
    # Shape: [batch_size, feature_num]
    embedding_2 = model.encode(texts_2, convert_to_tensor=True)

    # Cosine similarity
    # Shape: [batch_size]
    style_similarity_score = F.cosine_similarity(embedding_1, embedding_2, dim=1)

    # Save data
    style_similarity_scores += style_similarity_score.detach().cpu().tolist()
    all_text_1 += list(texts_1)
    all_text_2 += list(texts_2)
    author_scores += labels.tolist()

### Create dataframe

In [ ]:
# Create new DataFrame
df_output = pd.DataFrame({
    "text_1": all_text_1,
    "text_2": all_text_2,
    "author_score": author_scores, 
    "style_similarity_score": style_similarity_scores
})

### Create output csv

In [ ]:
# Save CSV
df_output.to_csv(OUTPUT_CSV_PATH, index=False)
print(f"Final training data saved to {OUTPUT_CSV_PATH}")